# PythonFMU3 in a notebook

Write an FMI 3.0 model in a cell, check it, simulate it, explore its parameters and build the FMU, all without leaving the notebook.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StephenSmith25/PythonFMU3/blob/master/examples/notebooks/getting_started.ipynb)

In [ ]:
# In Google Colab (or a fresh environment), install PythonFMU3 first:
# %pip install -q "pythonfmu3[notebook]"
%load_ext pythonfmu3

## Write a model

A water tank: water flows in at `q_in` and drains through a valve, so the outflow grows with the square root of the level. The `%%fmu` cell saves the class as `tank.py`, checks it and shows the FMU block and its variables. The `pythonfmu3` import is added for you if the cell doesn't have one.

In [ ]:
%%fmu
import math


class Tank(Fmi3Slave):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.author = "PythonFMU3 notebook example"
        self.description = "A tank draining through a valve"

        self.time = 0.0
        self.q_in = 0.05        # m3/s
        self.area = 2.0         # m2
        self.valve = 0.04       # m2.5/s, outflow = valve * sqrt(level)
        self.level = 1.0        # m
        self.q_out = self.valve * math.sqrt(self.level)

        self.register_units([Unit(name="m", m=1), Unit(name="m2", m=2), Unit(name="m3/s", m=3, s=-1)])
        self.register_variable(Float64("time", causality=Fmi3Causality.independent,
                                       variability=Fmi3Variability.continuous))
        self.register_variable(Float64("q_in", causality=Fmi3Causality.input, start=0.05, unit="m3/s"))
        self.register_variable(Float64("area", causality=Fmi3Causality.parameter,
                                       variability=Fmi3Variability.fixed, start=2.0, unit="m2"))
        self.register_variable(Float64("valve", causality=Fmi3Causality.parameter,
                                       variability=Fmi3Variability.tunable, start=0.04))
        self.register_variable(Float64("level", causality=Fmi3Causality.output, unit="m"))
        self.register_variable(Float64("q_out", causality=Fmi3Causality.output, unit="m3/s"))

    def do_step(self, current_time, step_size):
        self.q_out = self.valve * math.sqrt(max(self.level, 0.0))
        self.level = max(self.level + (self.q_in - self.q_out) / self.area * step_size, 0.0)
        self.time = current_time + step_size
        return True

## Simulate

`Tank` is now a model handle. `simulate()` runs the class directly and returns a pandas DataFrame indexed by time, so plotting and analysis work as usual.

In [ ]:
res = Tank.simulate(stop_time=300, start_values={"q_in": 0.05})
res.plot(y="level", ylabel="level [m]", grid=True);

## Explore a parameter

Because results are DataFrames, a parameter study is a loop. The level settles where inflow equals outflow, at (q_in / valve)².

In [ ]:
import pandas as pd

sweep = pd.DataFrame({
    f"valve = {v}": Tank.simulate(stop_time=1500, start_values={"valve": v})["level"]
    for v in (0.03, 0.04, 0.05, 0.07)
})
sweep.plot(ylabel="level [m]", grid=True);

## Build the FMU

`build()` writes `Tank.fmu` and shows a summary of it: interfaces, platforms with binaries and variables. Any simulation tool that imports FMI 3.0 FMUs can use it.

In [ ]:
fmu = Tank.build()
fmu

To check the built FMU in an importer, simulate it with FMPy (`pip install fmpy`) or fmusim:

```python
fmu.simulate(stop_time=300, start_values={"q_in": 0.05})
```

## Inspect any FMU

`FMU` works with any FMI 2.0 or 3.0 FMU, from any tool. It only reads the file.

In [ ]:
from pythonfmu3.notebook import FMU

FMU("Tank.fmu")